# Nền tảng 9 — Sau pretrain: SFT, DPO/GRPO, LoRA, và đánh giá LLM

Project dừng ở **pretrain**: huấn luyện model dự đoán token tiếp theo trên văn bản tiếng Việt, rồi đo bpc. Đó là
lựa chọn đúng cho câu hỏi nghiên cứu (so tokenizer) và cho ngân sách (36 giờ GPU). Nhưng mọi model bạn dùng hàng
ngày còn đi qua vài giai đoạn nữa, và notebook này dựng lại chúng ở mức đủ để bạn đọc hiểu tài liệu và biết khi
nào chúng áp dụng được.

Bốn chủ đề, mỗi chủ đề có một cài đặt đồ chơi chạy được:

1. **SFT** (supervised fine-tuning) — dạy model trả lời theo định dạng hội thoại; mấu chốt là **che loss**.
2. **RLHF → DPO → GRPO** — dạy model theo *ưu tiên của con người*; DPO là dạng đóng bỏ qua reward model.
3. **LoRA** — tinh chỉnh chỉ vài phần trăm tham số.
4. **Đánh giá LLM** — vì sao đo một model hội thoại khó hơn hẳn đo bpc, và những cái bẫy quen thuộc.

Cuối cùng là phần trả lời thẳng: **model d6/d8/d10 của project có làm được các bước này không.**

In [ ]:
import json
import math
import sys
from pathlib import Path

import torch
import torch.nn.functional as F

ROOT = Path.cwd()
while not (ROOT / "pixi.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "third_party" / "nanochat"))
torch.manual_seed(0)

from vitok.tokenizer_spec import SPECIAL_TOKENS

print("special token của nanochat (đã nằm sẵn trong tokenizer của project):")
for t in SPECIAL_TOKENS:
    print("  ", t)

## 1. Ba giai đoạn và thứ tự của chúng

| Giai đoạn | Dữ liệu | Hàm mục tiêu | Model học được gì |
|---|---|---|---|
| **Pretrain** | văn bản thô, hàng tỷ token | cross-entropy trên mọi token | ngôn ngữ, sự kiện, văn phong |
| **SFT** | hội thoại (câu hỏi → trả lời mẫu), hàng chục nghìn | cross-entropy **chỉ trên phần trả lời** | định dạng trả lời, làm theo yêu cầu |
| **Căn chỉnh** (RLHF/DPO/GRPO) | so sánh ưu tiên hoặc điểm thưởng | tối đa hoá phần thưởng, có ràng buộc | chọn giữa các câu trả lời đều hợp lệ |

Điểm dễ hiểu nhầm: SFT và RLHF **không dạy kiến thức mới**. Chúng định hình cách dùng những gì pretrain đã học.
Đó là lý do một model pretrain yếu không thể cứu bằng SFT — và cũng là lý do project đo ở tầng pretrain khi muốn
so tokenizer.

## 2. SFT và kỹ thuật che loss

Dữ liệu SFT là hội thoại, được ghép thành một chuỗi token bằng các special token đánh dấu vai:

```
<|bos|><|user_start|>Thủ đô Việt Nam là gì?<|user_end|><|assistant_start|>Hà Nội.<|assistant_end|>
```

Nếu huấn luyện cross-entropy trên **mọi** vị trí, model cũng học cách sinh ra *câu hỏi của người dùng* — không
phải điều ta muốn, và phần đó chiếm phần lớn token. Nên dùng **mặt nạ loss**: chỉ các token thuộc phần trả lời
mới đóng góp vào loss, phần còn lại đặt nhãn `-1` (giá trị `ignore_index` của `F.cross_entropy`).

Đúng như trong `chat_sft.py` của nanochat:

```python
# mask=1 for assistant completions, mask=0 for user prompts, BOS, special tokens, tool outputs
targets[mask_targets == 0] = -1
```

Cell dưới cho thấy che loss đổi giá trị loss bao nhiêu, và quan trọng hơn là đổi **gradient đi đâu**.

In [ ]:
V = 64
hoi_thoai = {"nguoi_dung": [5, 6, 7, 8, 9], "tra_loi": [20, 21, 22]}
ids = torch.tensor([[1] + hoi_thoai["nguoi_dung"] + [2] + hoi_thoai["tra_loi"] + [3]])   # 1,2,3 = special
mask = torch.tensor([[0] * (1 + len(hoi_thoai["nguoi_dung"]) + 1) + [1] * len(hoi_thoai["tra_loi"]) + [0]])
print("chuỗi :", ids.tolist()[0])
print("mặt nạ:", mask.tolist()[0], " (1 = tính loss)")

logits = torch.randn(1, ids.size(1), V, requires_grad=True)
x, y = logits[:, :-1], ids[:, 1:].clone()
y_che = y.clone()
y_che[mask[:, 1:] == 0] = -1                              # -1 = bỏ qua

loss_het = F.cross_entropy(x.reshape(-1, V), y.reshape(-1))
loss_che = F.cross_entropy(x.reshape(-1, V), y_che.reshape(-1), ignore_index=-1)
print(f"\nloss trên mọi vị trí : {loss_het.item():.4f}  (trung bình trên {y.numel()} token)")
print(f"loss chỉ phần trả lời: {loss_che.item():.4f}  (trung bình trên {(y_che != -1).sum().item()} token)")

g = torch.autograd.grad(loss_che, logits, retain_graph=True)[0]
print(f"\ngradient khác 0 tại các vị trí: {(g.abs().sum(-1)[0] > 0).nonzero().flatten().tolist()}")
print("=> đúng các vị trí dự đoán ra token của phần trả lời, không có vị trí nào khác.")

## 3. Từ RLHF tới DPO

### 3.1. Bài toán

Sau SFT, model đã trả lời đúng định dạng, nhưng giữa nhiều câu trả lời **đều hợp lệ** thì cái nào tốt hơn? Không
có nhãn đúng/sai, chỉ có **ưu tiên**: cho hai câu trả lời $y_w$ (được chọn) và $y_l$ (bị loại) cho cùng câu hỏi
$x$, con người nói thích cái nào hơn.

**RLHF cổ điển** làm ba bước:

1. Huấn luyện **reward model** $r_\phi(x,y)$ từ dữ liệu ưu tiên, theo mô hình Bradley–Terry:

$$P(y_w \succ y_l \mid x) = \sigma\big(r(x,y_w) - r(x,y_l)\big)$$

**Ký hiệu mới:** $\succ$ — "được ưa hơn"; $\sigma(u) = \frac{1}{1 + e^{-u}}$ — hàm sigmoid.

2. Tối ưu policy bằng RL (PPO) để tối đa reward, kèm phạt KL giữ policy gần model SFT:

$$\max_{\pi} \ \mathbb{E}_{y\sim\pi}\big[r(x,y)\big] - \beta\, D_{\mathrm{KL}}\big(\pi \,\|\, \pi_{\text{ref}}\big)$$

**Ký hiệu mới**
- $\pi$ — policy (model đang tối ưu); $\pi_{\text{ref}}$ — model tham chiếu, giữ cố định
- $\beta$ — hệ số phạt KL: lớn thì giữ policy gần $\pi_{\text{ref}}$

Phạt KL là phần không thể bỏ: không có nó, policy sẽ trôi tới những chuỗi lạ khiến reward model cho điểm cao mà
văn bản thì vô nghĩa — gọi là *reward hacking*.

### 3.2. Mẹo của DPO

Bài toán tối ưu ở (2) có **nghiệm dạng đóng**:

$$\pi^{*}(y\mid x) = \frac{1}{Z(x)}\,\pi_{\text{ref}}(y\mid x)\exp\!\left(\frac{r(x,y)}{\beta}\right)$$

**Ký hiệu mới:** $\pi^{*}$ — policy tối ưu; $Z(x)$ — hằng số làm tổng xác suất bằng 1 (tổng qua mọi câu trả lời,
không tính được trong thực tế).

Đảo ngược lại để biểu diễn reward theo policy:

$$r(x,y) = \beta\log\frac{\pi^{*}(y\mid x)}{\pi_{\text{ref}}(y\mid x)} + \beta\log Z(x)$$

(Lấy $\log$ hai vế rồi nhân với $\beta$.)

Thay vào mô hình Bradley–Terry, **$Z(x)$ triệt tiêu** vì nó chỉ phụ thuộc $x$, xuất hiện ở cả hai vế của hiệu.
Kết quả là một hàm mất mát học có giám sát thẳng trên dữ liệu ưu tiên, không cần reward model và không cần RL:

$$\mathcal{L}_{\mathrm{DPO}} = -\log\sigma\!\left(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)$$

**Ký hiệu mới:** $\pi_\theta(y|x)$ — xác suất policy đang học gán cho **cả câu trả lời** $y$ (tích xác suất các
token). Mỗi số hạng $\beta\log\frac{\pi_\theta}{\pi_{\text{ref}}}$ là reward ngầm của câu trả lời đó.

Đọc công thức: đại lượng trong ngoặc là "model hiện tại thích $y_w$ hơn $\pi_{\text{ref}}$ bao nhiêu, trừ đi mức
đó cho $y_l$". Tối thiểu hoá $-\log\sigma$ của nó nghĩa là đẩy hiệu này lên cao.

In [ ]:
def dpo_loss(logp_w, logp_l, ref_w, ref_l, beta=0.1):
    return -F.logsigmoid(beta * ((logp_w - ref_w) - (logp_l - ref_l)))


ref_w, ref_l = torch.tensor(-10.0), torch.tensor(-10.0)      # model tham chiếu trung lập
print(" logπ(y_w) | logπ(y_l) | margin | loss DPO (β=0.1)")
for lw, ll in ((-10.0, -10.0), (-9.0, -11.0), (-5.0, -15.0), (-11.0, -9.0)):
    lw_t, ll_t = torch.tensor(lw), torch.tensor(ll)
    margin = (lw_t - ref_w) - (ll_t - ref_l)
    print(f" {lw:9.1f} | {ll:9.1f} | {margin:+6.1f} | {dpo_loss(lw_t, ll_t, ref_w, ref_l).item():.4f}")

print("\nvai trò của β (cùng một margin = +4):")
for beta in (0.01, 0.1, 0.5, 1.0):
    print(f"  β={beta:4.2f} -> loss {dpo_loss(torch.tensor(-8.0), torch.tensor(-12.0), ref_w, ref_l, beta).item():.4f}")
print("β nhỏ -> cùng margin vẫn còn loss lớn -> gradient tiếp tục đẩy margin lên -> policy đi XA π_ref hơn.")
print("β lớn -> margin nhỏ đã đủ làm loss gần 0 -> policy dừng GẦN π_ref.")
print("Khớp với mục tiêu RLHF: β là hệ số của số hạng phạt KL, nên β lớn = ràng buộc chặt, β nhỏ = ràng buộc lỏng.")

In [ ]:
# huấn luyện thật một "policy" đồ chơi: phân phối trên 4 câu trả lời
logits_pi = torch.zeros(4, requires_grad=True)
with torch.no_grad():
    ref_logp = torch.log_softmax(torch.zeros(4), -1).clone()     # π_ref = đều
uu_tien = [(0, 3), (0, 2), (1, 3), (0, 1)]                        # (được chọn, bị loại)
opt = torch.optim.SGD([logits_pi], lr=1.0)

for buoc in range(100):
    logp = torch.log_softmax(logits_pi, -1)
    loss = torch.stack([dpo_loss(logp[w], logp[l], ref_logp[w], ref_logp[l], beta=0.1)
                        for w, l in uu_tien]).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()

print("dữ liệu ưu tiên:", uu_tien, " (0 luôn thắng, 3 luôn thua)")
print("xác suất cuối  :", [f"{p:.3f}" for p in torch.softmax(logits_pi, -1).tolist()])
print("logπ - logπ_ref:", [f"{v:+.2f}" for v in (torch.log_softmax(logits_pi, -1) - ref_logp).tolist()])
print("=> câu trả lời 0 (thắng 3 lần) lên cao nhất, 3 (thua 2 lần) xuống thấp nhất, 1 và 2 ở giữa.")
print("   Chạy lâu hơn thì DPO tiếp tục đẩy về phân phối suy biến (mọi khối lượng dồn vào câu 0): loss")
print("   không có điểm dừng khi dữ liệu ưu tiên nhất quán — một hạn chế đã biết của DPO.")

### 3.3. GRPO — cách nanochat làm

GRPO (group relative policy optimization) chấm **từng câu trả lời** bằng một điểm thưởng — từ reward model
như DeepSeekMath, hoặc từ một hàm kiểm tra tự động như nanochat — thay vì dùng cặp ưu tiên: ví dụ bài toán
có đáp án đúng/sai kiểm tra được bằng máy. Với mỗi câu hỏi, sinh $G$ câu trả lời, chấm điểm từng cái, và dùng
**điểm tương đối trong nhóm** làm advantage:

$$A_i = \frac{r_i - \mathrm{mean}(r)}{\mathrm{std}(r)} \qquad\text{hoặc, như nanochat: } A_i = r_i - \mathrm{mean}(r)$$

**Ký hiệu mới:** $A_i$ — **advantage** của câu trả lời $i$: tốt hơn trung bình nhóm bao nhiêu.

Vì sao trừ trung bình: nó loại bỏ phần thưởng chung của câu hỏi (câu dễ thì mọi mẫu đều điểm cao) và chỉ giữ
phần **mẫu nào tốt hơn các mẫu khác của cùng câu hỏi**. Đây chính là vai trò của *baseline* trong policy
gradient, nhưng lấy từ nhóm thay vì từ một value network riêng — nên GRPO không cần mạng thứ hai.

`chat_rl.py` của nanochat ghi rõ hai lựa chọn đơn giản hoá: chạy **on-policy** nên bỏ tỷ lệ + clip của PPO, và
chỉ trừ trung bình chứ không chia độ lệch chuẩn. nanochat không nêu lý do; lập luận thường gặp (Liu et al. 2025,
"Dr. GRPO") là chia cho std làm các câu hỏi mà điểm của nhóm gần như đồng nhất — câu quá dễ hoặc quá khó — bị
khuếch đại lên cùng tầm với các câu có tín hiệu thật, tạo ra thiên lệch theo độ khó của câu hỏi.

In [ ]:
rewards = torch.tensor([[1.0, 1.0, 0.0, 1.0],       # câu dễ: hầu hết đúng
                        [0.0, 0.0, 1.0, 0.0],       # câu khó: hầu hết sai
                        [1.0, 1.0, 1.0, 1.0]])      # câu quá dễ: tất cả đúng

print(" nhóm | reward            | A = r - mean        | A = (r-mean)/std")
for i, r in enumerate(rewards):
    a_tru = r - r.mean()
    a_chuan = (r - r.mean()) / (r.std() + 1e-8)
    print(f"  {i}   | {r.tolist()} | {[f'{v:+.2f}' for v in a_tru.tolist()]} | {[f'{v:+.2f}' for v in a_chuan.tolist()]}")
print("\nnhóm 2: mọi mẫu đều đúng -> advantage bằng 0, không có tín hiệu học (đúng: không có gì để phân biệt).")
print("chia cho std: nhóm 0 và 1 (chỉ 1/4 mẫu khác biệt) bị phóng lên gấp đôi — mọi nhóm bị ép về cùng thang,")
print("bất kể tín hiệu trong nhóm mạnh hay yếu. Trừ mean giữ nguyên thang đo của reward.")

logp = torch.tensor([-2.0, -3.0, -1.5, -2.5], requires_grad=True)
A = rewards[0] - rewards[0].mean()
loss_pg = -(A * logp).mean()                         # policy gradient: đẩy logπ theo dấu advantage
loss_pg.backward()
print(f"\ngradient theo logπ: {logp.grad.tolist()}")
print("dấu âm = tăng xác suất (mẫu tốt hơn trung bình), dấu dương = giảm.")

## 4. LoRA: tinh chỉnh vài phần trăm tham số

Tinh chỉnh đầy đủ một model $N$ tham số cần lưu gradient và trạng thái optimizer cho cả $N$ — với AdamW là
khoảng **4 lần** bộ nhớ của model. LoRA (low-rank adaptation) dựa trên quan sát: thay đổi cần thiết khi thích nghi
sang một nhiệm vụ mới thường có **hạng thấp**. Nên thay vì học $\Delta W$ đầy đủ, học tích của hai ma trận gầy:

$$W' = W + \frac{\alpha}{r} BA, \qquad B \in \mathbb{R}^{d_{\text{out}} \times r},\ A \in \mathbb{R}^{r \times d_{\text{in}}}$$

**Ký hiệu mới**
- $d_{\text{in}}$, $d_{\text{out}}$ — số chiều vào/ra của ma trận $W$
- $\mathbb{R}^{m \times n}$ — tập ma trận số thực $m$ hàng $n$ cột
- $r$ — hạng của phần cập nhật; $\alpha$ — hệ số tỷ lệ (ở đây không phải reward và mức ý nghĩa)

với $r \ll \min(d_{\text{in}}, d_{\text{out}})$. $W$ **đóng băng**; chỉ $A, B$ được học. Số tham số huấn luyện
giảm từ $d_{\text{in}}d_{\text{out}}$ xuống $r(d_{\text{in}} + d_{\text{out}})$.

Khởi tạo: $A$ ngẫu nhiên, $B = 0$, nên lúc bắt đầu $BA = 0$ và model **đúng bằng** model gốc. Sau khi huấn luyện
xong có thể **gộp** $BA$ vào $W$, nên lúc suy luận không tốn thêm phép tính nào.

In [ ]:
class LoRALinear(torch.nn.Module):
    def __init__(self, W, r=4, alpha=8):
        super().__init__()
        self.W = torch.nn.Parameter(W.clone(), requires_grad=False)      # đóng băng
        d_out, d_in = W.shape
        self.A = torch.nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.B = torch.nn.Parameter(torch.zeros(d_out, r))               # B = 0 -> khởi đầu không đổi gì
        self.scale = alpha / r

    def forward(self, x):
        return x @ self.W.T + self.scale * (x @ self.A.T) @ self.B.T

    def gop(self):
        return self.W + self.scale * self.B @ self.A


d_in, d_out, r = 256, 256, 4
W = torch.randn(d_out, d_in) / math.sqrt(d_in)
lora = LoRALinear(W, r=r)

n_day_du = d_in * d_out
n_lora = r * (d_in + d_out)
print(f"tinh chỉnh đầy đủ: {n_day_du:,} tham số")
print(f"LoRA r={r}       : {n_lora:,} tham số ({n_lora / n_day_du:.2%})")

x = torch.randn(8, d_in)
print(f"\nlúc khởi tạo, đầu ra khác model gốc: {(lora(x) - x @ W.T).abs().max().item():.2e}  (bằng 0)")

muc_tieu = torch.randn(8, d_out)
opt = torch.optim.Adam([p for p in lora.parameters() if p.requires_grad], lr=0.01)
for _ in range(200):
    loss = F.mse_loss(lora(x), muc_tieu)
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f"sau 200 bước: loss {loss.item():.4f}")
print(f"W có đổi không: {(lora.W - W).abs().max().item():.2e}  (không — chỉ A và B học)")
print(f"gộp BA vào W rồi chạy lại: sai khác {(x @ lora.gop().T - lora(x)).abs().max().item():.2e}")

print(f"\náp lên model d8 của project ({74_514_666:,} tham số), LoRA r=8 trên mọi ma trận thân:")
n_than_lora = 8 * 8 * (512 + 512) * 4 + 8 * 8 * (512 + 2048) * 2      # xấp xỉ: 4 ma trận attn + 2 ma trận MLP
print(f"  ~{n_than_lora:,} tham số huấn luyện = {n_than_lora / 74_514_666:.2%} model")

## 5. Đánh giá một model hội thoại

Đo pretrain thì gọn: bpc trên văn bản giữ lại, một con số, so được trực tiếp (notebook 01–02). Đo một model trả
lời câu hỏi thì khó hơn nhiều, vì **không có một đáp án đúng duy nhất**. Bốn họ phương pháp, kèm điểm yếu:

| Cách đo | Ví dụ | Điểm yếu chính |
|---|---|---|
| Trắc nghiệm nhiều lựa chọn | MMLU, HellaSwag | nhạy với định dạng prompt và thứ tự lựa chọn; dễ nhiễm dữ liệu |
| Sinh văn bản + khớp chuỗi | GSM8K, HumanEval | chỉ dùng được khi có đáp án kiểm tra được bằng máy |
| Chấm bởi model khác | MT-Bench, AlpacaEval | model chấm thiên vị câu dài, thiên vị model cùng họ |
| So cặp bởi con người | Chatbot Arena | đắt, chậm; cần rất nhiều so sánh để có thống kê chắc |

Hai vấn đề đáng nhớ nhất:

**Nhiễm dữ liệu** (contamination): bộ kiểm tra đã lọt vào dữ liệu pretrain. Model "nhớ" đáp án thay vì suy luận,
và điểm cao trở nên vô nghĩa. Với model của project, khả năng này được kiểm soát vì tập test tách theo hash và
không dùng benchmark công khai nào.

**Nhạy với định dạng**: cùng câu hỏi, đổi thứ tự A/B/C/D là điểm đổi. Cell dưới mô phỏng mức nhiễu đó và nối
thẳng với phân tích power ở notebook 02: nếu nhiễu định dạng lớn hơn chênh lệch giữa hai model, phép đo không
kết luận được gì — giống hệt tình huống cặp tối thiểu chạm trần.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n_cau = 500
kha_nang = 0.62                                  # xác suất model "thật sự biết" một câu
biet = rng.random(n_cau) < kha_nang

print("mô phỏng: model đúng chắc chắn nếu biết; nếu không biết thì đoán, và việc đoán")
print("phụ thuộc thứ tự lựa chọn (thiên vị chọn A).\n")
print(" hoán vị | độ chính xác")
diem = []
for lan in range(5):
    thien_vi_A = rng.random(n_cau) < 0.35        # khi không biết, hay chọn A
    dap_an_la_A = rng.random(n_cau) < 0.25
    dung = biet | (~biet & thien_vi_A & dap_an_la_A) | (~biet & ~thien_vi_A & (rng.random(n_cau) < 0.25))
    diem.append(dung.mean())
    print(f"    {lan}    | {dung.mean():.3f}")
print(f"\nđộ lệch chuẩn giữa các hoán vị: {np.std(diem):.4f}")
print(f"=> hai model chênh nhau dưới ~{2 * np.std(diem):.3f} điểm thì không phân biệt được bằng bộ này,")
print("   dù mỗi con số riêng lẻ trông rất chính xác (3 chữ số thập phân).")

In [ ]:
# Bradley-Terry: từ kết quả so cặp ra thang điểm, đúng mô hình đứng sau Elo và Chatbot Arena
ket_qua = [(0, 1, 18, 7), (0, 2, 25, 5), (1, 2, 14, 12)]     # (A, B, số lần A thắng, số lần B thắng)
diem_bt = torch.zeros(3, requires_grad=True)
opt = torch.optim.Adam([diem_bt], lr=0.1)
for _ in range(2000):
    loss = sum(-(w * F.logsigmoid(diem_bt[a] - diem_bt[b]) + l * F.logsigmoid(diem_bt[b] - diem_bt[a]))
               for a, b, w, l in ket_qua)
    opt.zero_grad()
    loss.backward()
    opt.step()

with torch.no_grad():
    d = (diem_bt - diem_bt.mean())
print("điểm Bradley-Terry ước lượng:", [f"{v:+.3f}" for v in d.tolist()])
print("xác suất model 0 thắng model 2 theo mô hình:", f"{torch.sigmoid(d[0] - d[2]).item():.3f}")
print("quan sát thực tế                          :", f"{25 / 30:.3f}")
print("\n(Elo là đúng mô hình này, viết lại theo thang 400 điểm và cập nhật trực tuyến.)")

## 6. Model của project có dùng được các bước này không?

Câu trả lời thẳng, dựa trên số liệu ở notebook 03:

| | d6 | d8 | d10 |
|---|---|---|---|
| Tham số thân | 10,6M | 25,2M | 49,2M |
| Token pretrain | 250M | 500M | 1B |

Để so: GPT-2 small có 124M tham số (bài gốc ghi 117M) và được pretrain trên WebText khoảng 40GB văn bản, cỡ
10 tỷ token; các model hội thoại nhỏ được phát hành gần đây (ví dụ Qwen2.5-0.5B-Instruct) có từ 0,5B tham số trở
lên và được pretrain trên hàng nghìn tỷ token.

**SFT**: chạy được về mặt kỹ thuật — `chat_sft.py` sẽ hoạt động, loss sẽ giảm, model sẽ học định dạng
`<|user_start|>…<|assistant_start|>`. Nhưng nội dung trả lời sẽ không mạch lạc, vì pretrain chưa đủ để có kiến
thức và khả năng suy luận. Nói cách khác: **định dạng học được, nội dung thì không**.

**RLHF/DPO/GRPO**: về nguyên tắc chạy được, nhưng vô nghĩa ở quy mô này. Các phương pháp này *chọn giữa các câu
trả lời đều hợp lệ*; khi chưa có câu trả lời nào hợp lệ thì không có gì để chọn. GRPO còn cần model tự sinh được
lời giải đúng đôi khi để có advantage khác 0 — với d6 thì tỷ lệ đó gần như bằng 0 trên mọi nhiệm vụ có kiểm tra
được.

**LoRA**: không cần thiết. LoRA giải bài toán *không đủ bộ nhớ để tinh chỉnh đầy đủ*; model 74M tham số tinh
chỉnh đầy đủ vẫn vừa thoải mái trong 16GB (notebook 07 mục 6 đo đỉnh bộ nhớ pretrain chỉ 9GB).

**Đánh giá**: đây mới là phần đáng làm tiếp. Các benchmark tiếng Việt như VMLU hay ViGLUE giả định model đủ mạnh
để trả lời trắc nghiệm; với cỡ này, điểm sẽ nằm quanh mức đoán ngẫu nhiên, và theo đúng phân tích power ở
notebook 02 thì không phân biệt được các điều kiện. Phép đo hợp lý ở quy mô này vẫn là bpc cộng các phép thử
ngôn ngữ học có kiểm soát — đúng những gì project đang làm.

In [ ]:
print("để so sánh quy mô:\n")
print(" model                    | tham số      | token pretrain | tỷ lệ so với d10")
for ten, N, D in (("vitok d6", 10_616_832, 250e6), ("vitok d8", 25_165_824, 500e6),
                  ("vitok d10", 49_152_000, 1e9), ("GPT-2 small", 124e6, 10e9),
                  ("Llama 3 8B", 8e9, 15e12), ("Qwen 2.5 0.5B", 0.5e9, 18e12)):
    print(f" {ten:24s} | {N:12,.0f} | {D:14,.0f} | {N / 49_152_000:6.1f}× tham số, {D / 1e9:8.1f}× token")

print("\nFLOPs pretrain xấp xỉ (6ND):")
for ten, N, D in (("vitok d10", 49_152_000, 1e9), ("Llama 3 8B", 8e9, 15e12)):
    print(f"  {ten:12s}: {6 * N * D:.2e}")
print(f"  tỷ lệ: {6 * 8e9 * 15e12 / (6 * 49_152_000 * 1e9):,.0f} lần")

## 7. Tóm tắt

| Chủ đề | Công thức cốt lõi | Khi nào dùng |
|---|---|---|
| SFT | cross-entropy có che, `targets[mask == 0] = -1` | dạy định dạng hội thoại |
| Bradley–Terry | $P(y_w \succ y_l) = \sigma(r_w - r_l)$ | nền của reward model và của Elo |
| RLHF | $\max \mathbb{E}[r] - \beta D_{\mathrm{KL}}(\pi\|\pi_{\text{ref}})$ | cần reward model + PPO |
| DPO | $-\log\sigma(\beta[(\log\pi_\theta^w - \log\pi_{\text{ref}}^w) - (\log\pi_\theta^l - \log\pi_{\text{ref}}^l)])$ | có dữ liệu ưu tiên, muốn bỏ RL |
| GRPO | $A_i = r_i - \mathrm{mean}(r)$ trong nhóm $G$ mẫu | có hàm thưởng tự động |
| LoRA | $W + \frac{\alpha}{r}BA$, $B = 0$ lúc đầu | thiếu bộ nhớ để tinh chỉnh đầy đủ |

## 8. Câu hỏi tự kiểm

1. Vì sao SFT phải che loss ở phần câu hỏi của người dùng?
2. Trong mục tiêu RLHF, bỏ số hạng phạt KL đi thì hỏng ở đâu?
3. Vì sao $Z(x)$ triệt tiêu trong suy dẫn DPO?
4. $\beta$ trong DPO đóng vai trò gì, và nó tương ứng với đại lượng nào của RLHF?
5. Vì sao GRPO không cần value network?
6. Nhóm GRPO có 4 mẫu, reward $(1,1,1,1)$. Advantage bằng bao nhiêu, và model học được gì từ nhóm đó?
7. LoRA với $r = 8$ trên ma trận $2048 \times 2048$ tiết kiệm bao nhiêu phần trăm tham số huấn luyện?
8. Vì sao điểm trắc nghiệm của một model nhỏ trên VMLU lại không phân biệt được bốn điều kiện tokenizer của
   project? Trả lời bằng ngôn ngữ power ở notebook 02.

**Đáp án gợi ý**

1. Để model không học cách sinh ra câu hỏi; phần đó chiếm phần lớn token và không phải thứ ta muốn model tạo ra.
2. Policy trôi khỏi phân phối ngôn ngữ tự nhiên để khai thác lỗ hổng của reward model (*reward hacking*).
3. Vì $Z(x)$ chỉ phụ thuộc $x$, nên trong hiệu $r(x,y_w) - r(x,y_l)$ nó xuất hiện hai lần với dấu ngược nhau.
4. $\beta$ là hệ số phạt KL, chính là $\beta$ trong mục tiêu RLHF: $\beta$ **lớn** thì ràng buộc chặt vào
   $\pi_{\text{ref}}$, $\beta$ **nhỏ** thì policy được phép đi xa hơn.
5. Vì trung bình reward của nhóm đóng vai trò baseline, thay cho ước lượng giá trị trạng thái.
6. Advantage bằng 0 với mọi mẫu; không có tín hiệu học — đúng, vì nhóm không cho biết mẫu nào tốt hơn.
7. $8 \times (2048 + 2048) = 32\,768$ so với $2048^2 = 4{,}19$M, tức khoảng $0{,}78\%$.
8. Vì độ chính xác nằm quanh mức đoán ngẫu nhiên và nhiễu định dạng lớn hơn chênh lệch thật, nên sai số chuẩn
   của hiệu lớn hơn hiệu ứng nhiều lần — power gần như bằng 0, giống tình huống chạm trần của cặp tối thiểu.

**Nguồn đọc thêm**

- [Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155) — InstructGPT, mục 3.
- [Direct Preference Optimization](https://arxiv.org/abs/2305.18290) — mục 4 có toàn bộ suy dẫn ở trên.
- [DeepSeekMath](https://arxiv.org/abs/2402.03300) — GRPO được giới thiệu ở mục 4.
- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685) — mục 4.
- [QLoRA](https://arxiv.org/abs/2305.14314) — mục 3, LoRA trên model lượng tử hoá 4-bit, vừa máy 6GB.
- [Holistic Evaluation of Language Models (HELM)](https://arxiv.org/abs/2211.09110) — khảo sát cách đánh giá.
- [NLP Evaluation in trouble](https://arxiv.org/abs/2310.18018) — nhiễm dữ liệu benchmark.
- Công cụ để tự chạy: [TRL](https://huggingface.co/docs/trl) (SFT, DPO, GRPO),
  [PEFT](https://huggingface.co/docs/peft) (LoRA), [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness).
- `third_party/nanochat/scripts/chat_sft.py` và `chat_rl.py` — bản cài đặt gọn của SFT và GRPO.